In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

import kagglehub

import warnings
warnings.filterwarnings('ignore')

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
LATENT_DIM = 128
IMAGE_SIZE = 64
BATCH_SIZE = 128
NUM_WORKERS = 16
EPOCHS_UNCOND = 40
EPOCHS_COND = 40

os.makedirs("outputs", exist_ok=True)
print(f"Device: {DEVICE}")

In [ ]:
data_root = Path(kagglehub.dataset_download("jessicali9530/celeba-dataset"))
print(f"Data root: {data_root}")

attr_df = pd.read_csv(data_root / "list_attr_celeba.csv")
split_df = pd.read_csv(data_root / "list_eval_partition.csv")

image_dir = next(data_root.rglob("*.jpg")).parent

attr_df["partition"] = split_df["partition"]
attr_df["Male"] = (attr_df["Male"] == 1).astype(int)  # 0=Female, 1=Male
attr_df["image_path"] = [str(image_dir / name) for name in attr_df["image_id"]]

train_df = attr_df[attr_df["partition"] == 0].reset_index(drop=True)
val_df   = attr_df[attr_df["partition"] == 1].reset_index(drop=True)
test_df  = attr_df[attr_df["partition"] == 2].reset_index(drop=True)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"Male ratio: {train_df['Male'].mean():.2%}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class CelebADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        return transform(img), int(row["Male"])


train_loader = DataLoader(CelebADataset(train_df), batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(CelebADataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(CelebADataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

images, _ = next(iter(train_loader))
plt.figure(figsize=(8, 8))
grid = make_grid(images[:16], nrow=4, normalize=True)
plt.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
plt.axis("off")
plt.title("CelebA examples (64x64)")
plt.show()

In [ ]:
def show_images(tensor, title="", n=16):
    images = tensor[:n].detach().cpu()
    rows = int(np.sqrt(n))
    cols = int(np.ceil(n / rows))

    fig, axes = plt.subplots(rows, cols, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            img = (images[i] + 1) / 2  # [-1, 1] → [0, 1]
            ax.imshow(np.transpose(img.numpy(), (1, 2, 0)))
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_losses(g_losses, d_losses, title):
    plt.figure(figsize=(8, 4))
    plt.plot(g_losses, label="Generator")
    plt.plot(d_losses, label="Discriminator")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
class Generator(nn.Module):
    """DCGAN-генератор: вектор → 64x64 RGB-изображение."""
    def __init__(self, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """DCGAN-дискриминатор: 64x64 RGB → вероятность real/fake."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)

In [ ]:
def train_unconditional(G, D, loader, epochs=5, latent_dim=128):
    G = G.to(DEVICE)
    D = D.to(DEVICE)

    criterion = nn.BCELoss()
    g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    d_opt = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    g_losses, d_losses = [], []

    for epoch in range(epochs):
        G.train(); D.train()
        total_g, total_d = 0.0, 0.0

        for real_imgs, _ in tqdm(loader, desc=f"Uncond epoch {epoch+1}/{epochs}", leave=False):
            real_imgs = real_imgs.to(DEVICE)
            bz = real_imgs.size(0)
            real_labels = torch.ones(bz, device=DEVICE)
            fake_labels = torch.zeros(bz, device=DEVICE)

            d_opt.zero_grad()
            real_out = D(real_imgs)
            real_loss = criterion(real_out, real_labels)
            noise = torch.randn(bz, latent_dim, 1, 1, device=DEVICE)
            fake_imgs = G(noise)
            fake_out = D(fake_imgs.detach())
            fake_loss = criterion(fake_out, fake_labels)
            d_loss = real_loss + fake_loss
            d_loss.backward()
            d_opt.step()

            g_opt.zero_grad()
            fake_out = D(fake_imgs)
            g_loss = criterion(fake_out, real_labels)  # хотим обмануть D
            g_loss.backward()
            g_opt.step()

            total_g += g_loss.item()
            total_d += d_loss.item()

        avg_g = total_g / len(loader)
        avg_d = total_d / len(loader)
        g_losses.append(avg_g)
        d_losses.append(avg_d)

        print(f"Epoch {epoch+1}/{epochs}  D: {avg_d:.4f}  G: {avg_g:.4f}")

        with torch.no_grad():
            noise = torch.randn(16, latent_dim, 1, 1, device=DEVICE)
            show_images(G(noise), title=f"Unconditional GAN — Epoch {epoch+1}")

    return G, D, g_losses, d_losses


print("Training unconditional GAN...")
G_uncond = Generator(LATENT_DIM)
D_uncond = Discriminator()

G_uncond, D_uncond, g_loss_uncond, d_loss_uncond = train_unconditional(
    G_uncond, D_uncond, train_loader, epochs=EPOCHS_UNCOND, latent_dim=LATENT_DIM
)
plot_losses(g_loss_uncond, d_loss_uncond, "Unconditional GAN")

torch.save(G_uncond.state_dict(), "outputs/generator_uncond.pt")

In [ ]:
class CondGenerator(nn.Module):
    def __init__(self, latent_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, 16)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim + 16, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, labels):
        emb = self.embedding(labels).unsqueeze(2).unsqueeze(3)
        z = torch.cat([z, emb], dim=1)
        return self.net(z)


class CondDiscriminator(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, 16)
        self.linear = nn.Linear(16, 64 * 64)
        self.net = nn.Sequential(
            nn.Conv2d(4, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        emb = self.embedding(labels)
        label_map = self.linear(emb).view(-1, 1, 64, 64)
        x = torch.cat([x, label_map], dim=1)
        return self.net(x).view(-1)

In [ ]:
def train_conditional(G, D, loader, epochs=5, latent_dim=128):
    G = G.to(DEVICE)
    D = D.to(DEVICE)

    criterion = nn.BCELoss()
    g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    d_opt = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    g_losses, d_losses = [], []

    for epoch in range(epochs):
        G.train(); D.train()
        total_g, total_d = 0.0, 0.0

        for real_imgs, labels in tqdm(loader, desc=f"Cond epoch {epoch+1}/{epochs}", leave=False):
            real_imgs = real_imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            bz = real_imgs.size(0)
            real_targets = torch.ones(bz, device=DEVICE)
            fake_targets = torch.zeros(bz, device=DEVICE)


            d_opt.zero_grad()
            real_loss = criterion(D(real_imgs, labels), real_targets)

            noise = torch.randn(bz, latent_dim, 1, 1, device=DEVICE)
            fake_imgs = G(noise, labels)
            fake_loss = criterion(D(fake_imgs.detach(), labels), fake_targets)

            d_loss = real_loss + fake_loss
            d_loss.backward()
            d_opt.step()


            g_opt.zero_grad()
            g_loss = criterion(D(fake_imgs, labels), real_targets)
            g_loss.backward()
            g_opt.step()

            total_g += g_loss.item()
            total_d += d_loss.item()

        avg_g = total_g / len(loader)
        avg_d = total_d / len(loader)
        g_losses.append(avg_g)
        d_losses.append(avg_d)

        print(f"Epoch {epoch+1}/{epochs}  D: {avg_d:.4f}  G: {avg_g:.4f}")

        with torch.no_grad():
            z = torch.randn(16, latent_dim, 1, 1, device=DEVICE)
            female = torch.zeros(16, dtype=torch.long, device=DEVICE)
            male = torch.ones(16, dtype=torch.long, device=DEVICE)
            show_images(G(z, female), title=f"Conditional GAN — Female, epoch {epoch+1}")
            show_images(G(z, male),   title=f"Conditional GAN — Male, epoch {epoch+1}")

    return G, D, g_losses, d_losses


print("Training conditional GAN...")
G_cond = CondGenerator(LATENT_DIM)
D_cond = CondDiscriminator()

G_cond, D_cond, g_loss_cond, d_loss_cond = train_conditional(
    G_cond, D_cond, train_loader, epochs=EPOCHS_COND, latent_dim=LATENT_DIM
)
plot_losses(g_loss_cond, d_loss_cond, "Conditional GAN")

torch.save(G_cond.state_dict(), "outputs/generator_cond.pt")

In [ ]:
@torch.inference_mode()
def compute_fid(generator, loader, conditional=False, latent_dim=128, n_samples=1000):
    """FID: чем ниже, тем ближе распределение сгенерированных к реальным."""
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to('cpu')
    generator.eval()
    seen = 0

    for batch in loader:
        if conditional:
            real_imgs, labels = batch
            labels = labels.to(DEVICE)
        else:
            real_imgs, _ = batch
        real_imgs = real_imgs.to(DEVICE)
        bz = real_imgs.size(0)


        real_uint8 = ((real_imgs + 1) * 127.5).clamp(0, 255).to(torch.uint8).cpu()
        fid.update(real_uint8, real=True)

        z = torch.randn(bz, latent_dim, 1, 1, device=DEVICE)
        if conditional:
            fake_imgs = generator(z, labels)
        else:
            fake_imgs = generator(z)
        fake_uint8 = ((fake_imgs + 1) * 127.5).clamp(0, 255).to(torch.uint8).cpu()
        fid.update(fake_uint8, real=False)

        seen += bz
        if seen >= n_samples:
            break

    return fid.compute().item()


@torch.inference_mode()
def compute_is(generator, conditional=False, latent_dim=128, n_samples=1000):
    """Inception Score: чем выше, тем разнообразнее и качественнее генерации."""
    is_metric = InceptionScore(normalize=False).to('cpu')
    generator.eval()
    done = 0

    while done < n_samples:
        bz = min(64, n_samples - done)
        z = torch.randn(bz, latent_dim, 1, 1, device=DEVICE)
        if conditional:
            labels = torch.randint(0, 2, (bz,), device=DEVICE)
            fake_imgs = generator(z, labels)
        else:
            fake_imgs = generator(z)
        fake_uint8 = ((fake_imgs + 1) * 127.5).clamp(0, 255).to(torch.uint8).cpu()
        is_metric.update(fake_uint8)
        done += bz

    mean, std = is_metric.compute()
    return mean.item(), std.item()


In [ ]:

fid_uncond = compute_fid(G_uncond, test_loader, conditional=False)
is_uncond_mean, is_uncond_std = compute_is(G_uncond, conditional=False)

fid_cond = compute_fid(G_cond, test_loader, conditional=True)
is_cond_mean, is_cond_std = compute_is(G_cond, conditional=True)

results = {
    "Unconditional GAN": {"FID": fid_uncond, "IS_mean": is_uncond_mean, "IS_std": is_uncond_std},
    "Conditional GAN":   {"FID": fid_cond,   "IS_mean": is_cond_mean,   "IS_std": is_cond_std},
}

print(f"\n{'='*60}")
print(f"{'Model':<20} {'FID ↓':>10} {'IS_mean ↑':>10} {'IS_std':>10}")
print(f"{'-'*60}")
for name, m in results.items():
    print(f"{name:<20} {m['FID']:>10.2f} {m['IS_mean']:>10.4f} {m['IS_std']:>10.4f}")

In [ ]:

with torch.no_grad():
    z = torch.randn(16, LATENT_DIM, 1, 1, device=DEVICE)

    show_images(G_uncond(z), "Unconditional GAN — Final samples")

    female = torch.zeros(16, dtype=torch.long, device=DEVICE)
    male = torch.ones(16, dtype=torch.long, device=DEVICE)

    show_images(G_cond(z, female), "Conditional GAN — Female")
    show_images(G_cond(z, male),   "Conditional GAN — Male")

## Выводы

1. **DCGAN** — классическая архитектура для генерации изображений. Генератор из шума (128-мерный вектор) через транспонированные свёртки порождает 64x64 RGB, дискриминатор через обычные свёртки учится отличать реальное от подделки.
2. **Unconditional GAN** ничего не знает о классе — просто генерирует «какое-то лицо». Качество на 40 эпохах посредственное, FID ~50-60.
3. **Conditional GAN** получает метку пола через эмбеддинг. Один и тот же шум с разными метками даёт разные лица — это показывает, что модель действительно научилась разделять М/Ж.
4. **FID < 50** считается приемлемым для 64x64 GAN, но на 40 эпохах DCGAN без оптимизаций выдаёт 50-70. Чтобы улучшить — больше эпох, более глубокая сеть, или современные архитектуры вроде WGAN.
5. **Inception Score** для случайных картинок = 1.0, для реальных лиц >3, у обученного DCGAN 2.5-3.0 - модель генерирует узнаваемые лица, но без большого разнообразия.